# Cohort Data Creation

In [ ]:
import pandas as pd
from LabData.DataLoaders.GutMBLoader import GutMBLoader
from LabData.DataLoaders.SubjectLoader import SubjectLoader
from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
from LabData.DataLoaders.DietaryInterventionLoader import DietaryInterventionLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler


In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'
species = 'segal_species'

# Study configuration
study_ids = ["PNP3"]  # [15] for Australian cohort, ['PNP3'] for PNP3
# Map study_ids to study name for file naming
study_name_map = {15: 'AU15', 'PNP3': 'PNP3'}
study_name = study_name_map.get(study_ids[0] if isinstance(study_ids[0], int) else study_ids[0], f'study_{study_ids[0]}')

min_col_present_frac = 0.05

In [ ]:
diet_mb_10k = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features_10k, all_features_10k, targets_10k = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler_10k = pickle.load(scaler_file)
diet_mb_10k

In [ ]:
def explore_columns(df):
    for column in df.columns:
        print(column)
        print(df[column].value_counts())

# study_ids is defined in cell 2
subjects_dl = SubjectLoader()
subjects_data = subjects_dl.get_data(groupby_reg='first', study_ids=study_ids)
subjects_df = subjects_data.df
print(subjects_df)

## Load Microbiome Data

In [ ]:
gut_bacteria = GutMBLoader().get_data(SPECIES, subjects_df=subjects_df, study_ids=study_ids,
                                # groupby_reg='first', 
                                genotek_vals=[1], min_col_val=1e-4, take_log=True)
gut_bacteria_df = gut_bacteria.df.dropna(axis=1, how='all')
gut_bacteria_df.columns = gut_bacteria_df.columns.str.replace('s__', '')
# with open(home_path + f'data/{species}/my_lists.pkl', 'rb') as file:
#         loaded_lists = pickle.load(file)
# base_features, all_diet_features, targets = loaded_lists
gut_bacteria_df = gut_bacteria_df[targets_10k]



gut_bacteria_df.head(3)

In [ ]:
gut_bacteria_df.shape

In [ ]:
gut_bacteria_df

In [ ]:
gut_bacteria.df_metadata

In [ ]:
gut_bacteria_df = gut_bacteria_df.join(gut_bacteria.df_metadata[['RegistrationCode', 'Date']]).set_index(['RegistrationCode', 'Date'])
gut_bacteria_df.tail(20)

In [ ]:
# Filter people that don't have two tests
gut_bacteria_df = gut_bacteria_df.groupby(level=0).filter(lambda x: x.index.get_level_values('Date').nunique() == 2).sort_index()

gut_bacteria_df

In [ ]:
# Normalize by row.

# Step 1: Convert to normal scale
gut_bacteria_df_normal = 10 ** gut_bacteria_df

# Step 2: Mask of values that are NOT 0.0001
mask = gut_bacteria_df_normal != 0.0001

# Step 3: Row-wise sum of the non-0.0001 values
non_floor_sum = gut_bacteria_df_normal.where(mask).sum(axis=1)

# Step 4: Normalize ONLY the non-0.0001 values, keep 0.0001 unchanged
gut_bacteria_df_normal = gut_bacteria_df_normal.where(~mask, gut_bacteria_df_normal.div(non_floor_sum, axis=0))

# Step 5: Convert back to log10
gut_bacteria_df_log = np.log10(gut_bacteria_df_normal)
gut_bacteria_df = gut_bacteria_df_log
gut_bacteria_df

In [ ]:
row_number = gut_bacteria_df.groupby(level=0).cumcount()

# Split the DataFrame into baseline_mb and intervention_mb based on the row number
baseline_mb = gut_bacteria_df[row_number == 0]
intervention_mb = gut_bacteria_df[row_number == 1]

baseline_mb = baseline_mb.reset_index(level=[1], drop=True)
intervention_mb = intervention_mb.reset_index(level=[1], drop=True)

In [ ]:
gut_bacteria_df_col = gut_bacteria.df_columns_metadata
# View 5 most common bacteria
gut_bacteria_df_col[gut_bacteria_df_col['Unnamed: 0'].isin(["Rep_485", "Rep_609", "Rep_477", "Rep_449", "Rep_231"])]

In [ ]:
gut_bacteria_df_col.to_pickle(home_path + f"data/mb_names_{study_name.lower()}.pkl")

In [ ]:
gut_bacteria_df_meta = gut_bacteria.df_metadata
print(gut_bacteria_df_meta.head())
# Check that there's only one mb test per person
gut_bacteria_df_meta.RegistrationCode.value_counts()

### Alpha diversity targets

In [ ]:
# Richness
def richness(row):
    filtered = row[row > -4]
    return len(filtered)

baseline_mb['Richness'] = baseline_mb.apply(richness, axis=1)
intervention_mb['Richness'] = intervention_mb.apply(richness, axis=1)
baseline_mb

In [ ]:
# Shannon Diversity
def shannon(row):
    filtered = row[row > -4]
    filtered = filtered.drop("Richness")
    
    rel_abundance = 10 ** filtered
    ln_rel_abundance = np.log(rel_abundance.replace(0, 1))
    product = rel_abundance * ln_rel_abundance
    ans = -1 * product.sum()
    return round(float(ans), 2)

baseline_mb['Shannon_diversity'] = baseline_mb.apply(shannon, axis=1)
intervention_mb['Shannon_diversity'] = intervention_mb.apply(shannon, axis=1)
intervention_mb

In [ ]:

if species == 'mpa_species':
    path_to_tree = '/net/mraid20/export/genie/Bin/frcfrc/segata.k21.2023-01-31.tree'
elif species == 'segal_species':
    path_to_tree = '/net/mraid20/export/genie/Bin/frcfrc/segal.k21.2023-01-10.tree'
else:
    raise ValueError(f"Unknown SPECIES: {species}")

with open(path_to_tree, 'r') as f:
    print(f.read(100))

In [ ]:
import pandas as pd
from skbio import TreeNode

# 1. Load the tree (if not already loaded)
tree = TreeNode.read(path_to_tree)

# 2. Create the Mapping Dictionary
# The user specified: 
#   - Key (Source): First column of dataframe (values like "Rep_33")
#   - Value (Target): The index of the dataframe (values like "fBin__14|gBin__27|sBin__33")
# We create a dictionary: {'Rep_33': 'fBin__14|gBin__27|sBin__33', ...}
name_mapping = pd.Series(
    gut_bacteria_df_col.index.values,       # The Target (Index)
    index=gut_bacteria_df_col.iloc[:, 0]    # The Source (First Column)
).to_dict()

# 3. Rename the Tree Tips
count_renamed = 0
for tip in tree.tips():
    # The current tip name is like "Rep_33.fa.gz"
    # We strip ".fa.gz" to get "Rep_33" so we can look it up in the dictionary
    clean_name = tip.name.replace(".fa.gz", "")
    
    # Check if this Rep ID exists in our mapping
    if clean_name in name_mapping:
        # Update the tip name to the full species string
        tip.name = name_mapping[clean_name]
        count_renamed += 1

print(f"Successfully renamed {count_renamed} tips.")

# Optional: Verify a few tips to make sure it worked
print("First 5 new tip names:")
for tip in list(tree.tips())[:5]:
    print(tip.name)

# 4. Now you can calculate Faith's PD using the dataframe with species index
# Ensure your abundance table columns also match these new tree tip names!

In [ ]:
# Iterate over all nodes (tips and internal nodes)
for node in tree.traverse():
    # If branch length is missing (None), set it to 0.0
    if node.length is None:
        node.length = 0.0

print("Tree patched: All missing branch lengths set to 0.0.")

In [ ]:
import pandas as pd
from skbio.diversity.alpha import faith_pd
import numpy as np

def get_faith_index(df, tree):
    """
    Calculates Faith's Phylogenetic Diversity.
    Fixes the 'Integer Truncation' bug by passing 1s instead of floats.
    """
    # 1. Drop metadata
    taxa_df = df.drop(columns=["Richness", "Shannon_diversity", "Faith_index"], errors='ignore')
    
    # 2. Alignment Check
    tree_tips = {tip.name for tip in tree.tips()}
    valid_cols = [c for c in taxa_df.columns if c in tree_tips]
    taxa_df = taxa_df[valid_cols]

    def calculate_row_pd(row):
        # 3. Filter for Presence
        # Keep species with log abundance > -4
        present_taxa = row[row > -4]
        
        if present_taxa.empty:
            return 0.0
        
        # 4. The Fix: Create a Fake Count Vector of Integers
        # We create a Series of 1s, indexed by the species names.
        # This tells faith_pd: "These species are definitely present."
        present_counts = pd.Series(1, index=present_taxa.index, dtype=int)
        
        try:
            return faith_pd(present_counts, present_taxa.index, tree)
        except ValueError:
            return float('nan')

    # Apply to every row
    return taxa_df.apply(calculate_row_pd, axis=1)

# Usage
baseline_mb['Faith_index'] = get_faith_index(baseline_mb, tree)
intervention_mb['Faith_index'] = get_faith_index(baseline_mb, tree)

# Check results (Should now be > 46.6)
print(baseline_mb[['Faith_index']].head())
print(intervention_mb[['Faith_index']].head())

In [ ]:
# Sanity check
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Calculate Simple Richness (Count of species > -4 log abundance)
# We drop the metadata columns first to ensure we only count bacteria

# 2. Plot
plt.figure(figsize=(8, 6))
plt.scatter(baseline_mb['Richness'], baseline_mb['Faith_index'], alpha=0.5)
plt.xlabel('Species Richness (Count > -4)')
plt.ylabel("Faith's Phylogenetic Diversity")
plt.title('Sanity Check: PD vs Richness')

# 3. Correlation Score
# corr, p_val = stats.spearmanr(gut_bacteria_df['Richness'], gut_bacteria_df['Faith_index'])
# print(f"Spearman Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(baseline_mb['Richness'], baseline_mb['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(baseline_mb['Shannon_diversity'], baseline_mb['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(baseline_mb['Richness'], baseline_mb['Shannon_diversity'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
plt.show()

In [ ]:
# Sanity check
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Calculate Simple Richness (Count of species > -4 log abundance)
# We drop the metadata columns first to ensure we only count bacteria

# 2. Plot
plt.figure(figsize=(8, 6))
plt.scatter(intervention_mb['Richness'], intervention_mb['Faith_index'], alpha=0.5)
plt.xlabel('Species Richness (Count > -4)')
plt.ylabel("Faith's Phylogenetic Diversity")
plt.title('Sanity Check: PD vs Richness')

# 3. Correlation Score
# corr, p_val = stats.spearmanr(gut_bacteria_df['Richness'], gut_bacteria_df['Faith_index'])
# print(f"Spearman Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(intervention_mb['Richness'], intervention_mb['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(intervention_mb['Shannon_diversity'], intervention_mb['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(intervention_mb['Richness'], intervention_mb['Shannon_diversity'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
plt.show()

In [ ]:
diversity_targets = ['Richness', 'Shannon_diversity', 'Faith_index']

## Load Diet Data

In [ ]:
# Configure data directory based on study_id
study_id_value = study_ids[0]
if study_id_value == 'PNP3':
    nastya_dir = '/net/mraid20/export/genie/LabData/Data/StudySpecificData/PNP3/data/'
    file_suffix = 'pnp3'
elif study_id_value == 15:
    nastya_dir = '/net/mraid20/export/genie/LabData/Data/StudySpecificData/AU15/data/'  # Update this path if different
    file_suffix = 'au15'
else:
    nastya_dir = f'/net/mraid20/export/genie/LabData/Data/StudySpecificData/{study_name}/data/'
    file_suffix = study_name.lower()

baseline_nutrients = pd.read_csv(nastya_dir + f'baseline_log_{file_suffix}.csv')
intervention_nutrients = pd.read_csv(nastya_dir + f'intervention_log_{file_suffix}.csv')
# baseline_mb = pd.read_csv(nastya_dir + 'baseline_species.csv')
food_cat_baseline = pd.read_csv(nastya_dir + 'food_categories_bl.csv')
food_cat_diff = pd.read_csv(nastya_dir + 'food_categories_diff.csv')
food_cat_int = pd.read_csv(nastya_dir + 'food_categories_int.csv')
log_grouped_diff = pd.read_csv(nastya_dir + 'log_grouped_diff.csv')
species_change_all = pd.read_csv(nastya_dir + 'species_change_all.csv')
species_change_final = pd.read_csv(nastya_dir + 'species_changes_final.csv')
intervention_foods = pd.read_csv(nastya_dir + f'intervention_foods_{file_suffix}.csv')
intervention_foods_all = pd.read_csv(nastya_dir + f'intervention_foods_{file_suffix}_all.csv')
intervention_foods_all_full = pd.read_csv(nastya_dir + f'intervention_foods_{file_suffix}_all_full.csv')
baseline_foods_all = pd.read_csv(nastya_dir + f'baseline_foods_{file_suffix}_all.csv')



In [ ]:
intervention_foods_all

In [ ]:
baseline_foods_all.columns

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl', 'rb') as file:
    food_shortnames = pickle.load(file)
food_shortnames

In [ ]:
baseline_foods_all

In [ ]:
baseline_foods_all = baseline_foods_all.set_index('RegistrationCode')
baseline_foods = baseline_foods_all[[col for col in all_features_10k if col in baseline_foods_all.columns]]

intervention_foods_all = intervention_foods_all.set_index('RegistrationCode')
intervention_foods = intervention_foods_all[[col for col in all_features_10k if col in intervention_foods_all.columns]]

In [ ]:
# Normalize by person to get mean % of daily calories
baseline_foods = baseline_foods.fillna(0)
baseline_foods = baseline_foods.div(baseline_foods.sum(axis=1), axis=0)
baseline_foods

intervention_foods = intervention_foods.fillna(0)
intervention_foods = intervention_foods.div(intervention_foods.sum(axis=1), axis=0)
intervention_foods

In [ ]:
# intervention_nutrients = intervention_nutrients.set_index('RegistrationCode').drop(['Main score'], axis=1)
# intervention_nutrients

In [ ]:
# # Dictionary with the mappings for renaming
# rename_dict = {
#     'Fructose': 'Fructose',
#     'caffeine_mg': 'Caffeine',
#     'calcium_mg': 'Calcium, Ca',
#     'carbohydrate_g': 'Carbohydrate, by difference',
#     'cholesterol_mg': 'Cholesterol',
#     'iron_mg': 'Iron, Fe',
#     'magnesium_mg': 'Magnesium, Mg',
#     'niacin_mg': 'Niacin',
#     'phosphorus_mg': 'Phosphorus, P',
#     'potassium_mg': 'Potassium, K',
#     'protein_g': 'Protein',
#     'raevitamina_ug': 'Vitamin A, RAE',
#     'riboflavin_mg': 'Riboflavin',
#     'sodium_mg': 'Sodium, Na',
#     'thiamin_mg': 'Thiamin',
#     'totaldietaryfiber_g': 'Fiber, total dietary',
#     'totalfolate_ug': 'Folate, total',
#     'totallipid_g': 'Total lipid (fat)',
#     'totalmonounsaturatedfattyacids_g': 'Fatty acids, total monounsaturated',
#     'totalpolyunsaturatedfattyacids_g': 'Fatty acids, total polyunsaturated',
#     'totalsaturatedfattyacids_g': 'Fatty acids, total saturated',
#     'vitaminb12_ug': 'Vitamin B-12',
#     'vitaminb6_mg': 'Vitamin B-6',
#     'vitaminc_mg': 'Vitamin C, total ascorbic acid',
#     'vitamind_iu': 'Vitamin D (D2 + D3)',
#     'vitamine_mg': 'vitamin_E',
#     'zinc_mg': 'Zinc, Zn'
# }

# # Rename the items in the list using the mapping
# intervention_nutrients.columns = [rename_dict.get(item, item) for item in intervention_nutrients.columns]

# intervention_nutrients


In [ ]:
# food_cat_baseline = food_cat_baseline.set_index('RegistrationCode')
# # Normalize by person to get mean % of daily calories
# food_cat_baseline = food_cat_baseline.div(food_cat_baseline.sum(axis=1), axis=0)
# food_cat_baseline

### BMI

In [ ]:
bml = BodyMeasuresLoader()
# bmld = bml.get_data(study_ids=study_ids, cols=['weight', 'height', 'bmr'])
bmld = bml.get_data(study_ids=study_ids)
bmldf = bmld.df
bmldf.info()

In [ ]:
bmldf

In [ ]:
import pandas as pd
import numpy as np

# 1. Reset index to make columns accessible
#    This moves 'RegistrationCode' and 'Date' from the index into regular columns
df = bmldf.reset_index()

# 2. Ensure Date is in datetime format (handles strings automatically)
df['Date'] = pd.to_datetime(df['Date'])

# 3. Sort by Code and Date to ensure order is chronological
df = df.sort_values(by=['RegistrationCode', 'Date'])

# --- Create Baseline DataFrame ---
# Group by ID and take the first (earliest) row
bmldf_baseline = df.groupby('RegistrationCode').first().reset_index()


# --- Create Intervention DataFrame ---

# A. Merge the baseline date back into the main dataframe for comparison
#    We create a temporary dataframe with just the ID and the Baseline Date
baseline_dates = bmldf_baseline[['RegistrationCode', 'Date']].rename(
    columns={'Date': 'baseline_date'}
)
df_merged = pd.merge(df, baseline_dates, on='RegistrationCode', how='inner')

# B. Calculate the difference in months
#    We calculate the difference in nanoseconds and divide by the average duration of a month
df_merged['months_diff'] = (df_merged['Date'] - df_merged['baseline_date']) / np.timedelta64(1, 'M')

# C. Filter: Keep rows strictly between 5 and 7 months after baseline
#    (You can adjust the boundaries or use >= if you want inclusive)
mask = (df_merged['months_diff'] >= 5) & (df_merged['months_diff'] <= 7)
df_intervention_candidates = df_merged[mask]

# D. Select the LATEST row from the candidates
#    (The user requested the "latest row" within that window)
bmldf_intervention = df_intervention_candidates.sort_values('Date').groupby('RegistrationCode').last().reset_index()


# --- Cleanup ---

# 4. Restore the original MultiIndex for both result dataframes
#    We drop the helper columns we created
bmldf_baseline = bmldf_baseline.set_index(['RegistrationCode', 'Date'])

# For intervention, we also need to drop the helper calculation columns
cols_to_drop = ['baseline_date', 'months_diff']
bmldf_intervention = bmldf_intervention.drop(columns=cols_to_drop).set_index(['RegistrationCode', 'Date'])

# Verify results
print(f"Baseline shape: {bmldf_baseline.shape}")
print(f"Intervention shape: {bmldf_intervention.shape}")

In [ ]:
bmldf_baseline["bmi"]

In [ ]:
bmldf_intervention["bmi"]

## Combine Dataframes

In [ ]:
# baseline_nutrients = baseline_nutrients.set_index('RegistrationCode')
# baseline_nutrients.columns = [rename_dict.get(item, item) for item in baseline_nutrients.columns]
# baseline_nutrients

In [ ]:
# diet_mb = baseline_foods.join(baseline_nutrients, how='inner')
# # diet_mb = diet_mb.dropna()
# diet_mb

# diet_mb_intervention = intervention_foods.join(intervention_nutrients, how='inner')
# diet_mb_intervention

In [ ]:
subjects_df = subjects_df.reset_index(level=[1], drop=True)

In [ ]:
subjects_df

In [ ]:
base_features = ["age", "gender"]
subjects_df.index = subjects_df.index.astype('int')
baseline_foods = baseline_foods.join(subjects_df[base_features])
baseline_foods = baseline_foods.rename(columns={'gender': 'sex'})
# diet_mb = diet_mb.reset_index(level=[1], drop=True)
# diet_mb = diet_mb.dropna()
baseline_foods


intervention_foods = intervention_foods.join(subjects_df[base_features])
intervention_foods = intervention_foods.rename(columns={'gender': 'sex'})
# diet_mb = diet_mb.reset_index(level=[1], drop=True)
# diet_mb = diet_mb.dropna()
intervention_foods

In [ ]:

# Drop the second level of index for bmldf_baseline and bmldf_intervention (leaving only RegistrationCode)
bmldf_baseline = bmldf_baseline.reset_index(level=1, drop=True)
bmldf_baseline.index = bmldf_baseline.index.astype('int')
baseline_foods = baseline_foods.join(bmldf_baseline["bmi"])
baseline_foods

bmldf_intervention = bmldf_intervention.reset_index(level=1, drop=True)
bmldf_intervention.index = bmldf_intervention.index.astype('int')
intervention_foods = intervention_foods.join(bmldf_intervention["bmi"])
intervention_foods


In [ ]:
pnp3_diet_features = baseline_foods.columns
pnp3_diet_features

In [ ]:
pnp3_10k_shared_features = [col for col in pnp3_diet_features if col in all_features_10k]
pnp3_10k_shared_features.remove('Fructose')
pnp3_10k_shared_features

In [ ]:
len(pnp3_10k_shared_features)

In [ ]:
baseline_foods = baseline_foods[pnp3_10k_shared_features]
intervention_foods = intervention_foods[pnp3_10k_shared_features]

In [ ]:
baseline_mb.index = baseline_mb.index.astype('int')
diet_mb_baseline = baseline_foods.join(baseline_mb)
diet_mb_baseline

In [ ]:
intervention_mb.index = intervention_mb.index.astype('int')
diet_mb_intervention = intervention_foods.join(intervention_mb)
diet_mb_intervention

In [ ]:
# from sklearn.preprocessing import StandardScaler

# with open(home_path + f'data/{SPECIES}/age_scaler.pkl', 'rb') as f:
#     age_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/mb_scaler.pkl', 'rb') as f:
#     mb_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/div_scaler.pkl', 'rb') as f:
#     div_scaler = pickle.load(f)

# diet_scaler = StandardScaler()
# diet_mb_10k_scaled = diet_scaler.fit_transform(diet_mb_10k[pnp3_10k_shared_features])

# # # Apply the scaler to the dataframe
# diet_mb_baseline.loc[:, pnp3_10k_shared_features] = diet_scaler.transform(diet_mb_baseline[pnp3_10k_shared_features])
# diet_mb_baseline.loc[:, ["age"]] = age_scaler.transform(diet_mb_baseline[["age"]])
# diet_mb_baseline.loc[:, targets_10k] = mb_scaler.transform(diet_mb_baseline[targets_10k])
# diet_mb_baseline.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_baseline[["Richness", "Shannon_diversity"]])
# # diet_mb_baseline[pnp3_10k_shared_features] = pd.DataFrame(scaler.transform(diet_mb_baseline[pnp3_10k_shared_features]), columns=diet_mb_baseline[pnp3_10k_shared_features].columns, index=diet_mb_baseline[pnp3_10k_shared_features].index)
# diet_mb_baseline.describe()

In [ ]:
# # # Apply the scaler to the dataframe
# diet_mb_intervention.loc[:, pnp3_10k_shared_features] = diet_scaler.transform(diet_mb_intervention[pnp3_10k_shared_features])
# diet_mb_intervention.loc[:, ["age"]] = age_scaler.transform(diet_mb_intervention[["age"]])
# diet_mb_intervention.loc[:, targets_10k] = mb_scaler.transform(diet_mb_intervention[targets_10k])
# diet_mb_intervention.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_intervention[["Richness", "Shannon_diversity"]])
# diet_mb_intervention.describe()

In [ ]:
# Count rows that have at least one NaN
nan_rows = diet_mb_intervention[diet_mb_intervention.isna().any(axis=1)]
num_nan_rows = len(nan_rows)

# Find which columns contain NaN values
nan_features = diet_mb_intervention.columns[diet_mb_intervention.isna().any()].tolist()

print(f"Number of rows with at least one NaN: {num_nan_rows}")
print("Features that contain NaN values:")
print(nan_features)


In [ ]:
print("\nNaN count per column:")
print(diet_mb_intervention.isna().sum()[diet_mb_intervention.isna().sum() > 0])
diet_mb_baseline = diet_mb_baseline.dropna(how='any')
diet_mb_intervention = diet_mb_intervention.dropna(how='any')


In [ ]:
# Making sure we have the exact same patients in each data
common_indices = diet_mb_baseline.index.intersection(diet_mb_intervention.index)
diet_mb_baseline = diet_mb_baseline.loc[common_indices].sort_index()
diet_mb_intervention = diet_mb_intervention.loc[common_indices].sort_index()
print(diet_mb_baseline.shape)
print(diet_mb_intervention.shape)

In [ ]:
print(diet_mb_baseline.shape)
print(diet_mb_intervention.shape)

In [ ]:
diet_mb_baseline.to_pickle(home_path + f'data/diet_mb_{study_name.lower()}_baseline.pkl')
with open(home_path + f'data/my_lists_{study_name.lower()}.pkl', 'wb') as file:
    pickle.dump([pnp3_10k_shared_features, targets_10k], file)
    
diet_mb_intervention.to_pickle(home_path + f'data/diet_mb_{study_name.lower()}_intervention.pkl')

In [ ]:
print(len(pnp3_10k_shared_features))
print(len(targets_10k))